# Analytical Queries Using Pandas read_sql_query

**Assignment:** Milestone 4 – Execute the analytical queries on the college PostgreSQL database
**Target Host:** 10.100.71.21
**Target Database / Schema:** 202618052 / DBMS_project
**Source Data:** Formula 1 World Championship (1950–2024)

## Objective

1. Writing the SQL queries for the analytical questions of Milestone 1
2. Submitting each query to the PostgreSQL server with the pandas `read_sql_query` method
3. Collecting the result of each query into a DataFrame object
4. Printing `head(10)` of each DataFrame on the console

## Queries

1. Career wins
2. Constructor dominance
3. All-time points leaders
4. Pit-stop efficiency
5. Busiest circuits
6. Lap-time evolution
7. Qualifying vs. race outcome
8. Winners by nationality
9. DNF / retirement analysis
10. Season points progression
11. Fastest-lap leaders (drivers)
12. Home-race advantage
13. Decade-wise team dominance
14. Grid position vs. points scored
15. Sprint vs. Grand Prix performance

## Note on the uploaded data

The data was uploaded in Milestone 3 with `to_sql` exactly as it appears in the preprocessed CSV files.
In the `results` table, the columns `position` and `rank` therefore contain the text `\N` for missing values,
and `points` is stored as a decimal (double precision) number. The queries handle this directly in SQL:

- `NULLIF(r.position, '\N')::int` turns `\N` into `NULL` and the rest into whole numbers,
- `r.rank = '1'` selects the fastest lap of a race,
- `::numeric` is added before `ROUND` where the value is a double precision number.

## Team Members

1. 202618016 - Krishika Lalwani
2. 202618013 - Devanshi vora
3. 202618052 - Lakshita Pagaria

In [1]:
import pandas as pd
from sqlalchemy import create_engine, URL, text

# Show all columns of a result on the console
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

## Create the SQLAlchemy connection

In [2]:
username = "202618052"
password = "Ljp_@2006#"
host = "10.100.71.21"
port = 5432
database = "202618052"
schema = "DBMS_project"

url = URL.create(
    "postgresql+psycopg",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database
)

# Create the engine (schema name in quotes so its capital letters are kept)
engine = create_engine(
    url,
    connect_args={
        "options": f'-csearch_path="{schema}",public'
    }
)

print(f"Connected to PostgreSQL database '{database}'")
print(f"Using schema '{schema}'")

Connected to PostgreSQL database '202618052'
Using schema 'DBMS_project'


In [3]:
try:
    with engine.connect() as connection:
        print("PostgreSQL connection successful!")
        print(connection.execute(text("SELECT version();")).scalar())
        print("Current schema:", connection.execute(text("SELECT current_schema();")).scalar())
except Exception as e:
    print("Connection failed:")
    print(e)

PostgreSQL connection successful!
PostgreSQL 10.20 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 4.8.5 20150623 (Red Hat 4.8.5-44), 64-bit
Current schema: DBMS_project


## check the tables before running the queries

Number of rows in each of the 12 tables of the `DBMS_project` schema.

In [4]:
table_list = [
    "circuits", "constructors", "drivers", "status", "races", "qualifying",
    "results", "sprint_results", "lap_times", "pit_stops",
    "driver_standings", "constructor_standings"
]

with engine.connect() as connection:
    for table_name in table_list:
        query = text(f'SELECT COUNT(*) FROM "{schema}"."{table_name}"')
        row_count = connection.execute(query).scalar()
        print(f"Table: {table_name} | Total rows: {row_count}")

Table: circuits | Total rows: 77
Table: constructors | Total rows: 212
Table: drivers | Total rows: 861
Table: status | Total rows: 139
Table: races | Total rows: 1125
Table: qualifying | Total rows: 10494
Table: results | Total rows: 26759
Table: sprint_results | Total rows: 360
Table: lap_times | Total rows: 589081
Table: pit_stops | Total rows: 11371
Table: driver_standings | Total rows: 34863
Table: constructor_standings | Total rows: 13391


## function to run a query

Each query is submitted with `pd.read_sql_query`, the result is collected into a DataFrame,
and `head(10)` of the DataFrame is printed on the console.

In [5]:
def run_query(sql_query):
    df = pd.read_sql_query(text(sql_query), engine)
    print(f"Rows returned: {len(df)}\n")
    print(df.head(10))
    return df

## Query 1: Career wins

**Question:** Which drivers have the highest number of career race wins, and how does the top 10 list look across all seasons?
**Tables involved:** drivers, results
**Key SQL / technique:** NATURAL JOIN, GROUP BY, COUNT, FILTER, ORDER BY / LIMIT

In [6]:
query_1 = r"""
SELECT d.forename || ' ' || d.surname                                         AS driver,
       d.nationality,
       COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int = 1)              AS wins,
       COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int <= 3)             AS podiums,
       COUNT(*)                                                               AS races_entered,
       ROUND(100.0 * COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int = 1)
             / COUNT(*), 1)                                                   AS win_pct
FROM results r
NATURAL JOIN drivers d                      
GROUP BY "driverId", d.forename, d.surname, d.nationality
ORDER BY wins DESC, podiums DESC
LIMIT 10;
"""

query_1_df = run_query(query_1)

Rows returned: 10

               driver nationality  wins  podiums  races_entered  win_pct
0      Lewis Hamilton     British   105      202            356     29.5
1  Michael Schumacher      German    91      155            308     29.5
2      Max Verstappen       Dutch    63      112            209     30.1
3    Sebastian Vettel      German    53      122            300     17.7
4         Alain Prost      French    51      106            202     25.2
5        Ayrton Senna   Brazilian    41       80            162     25.3
6     Fernando Alonso     Spanish    32      106            404      7.9
7       Nigel Mansell     British    31       59            192     16.1
8      Jackie Stewart     British    27       43            100     27.0
9          Niki Lauda    Austrian    25       54            174     14.4


## Query 2: Constructor dominance

**Question:** Which constructors have won the most Constructors' Championships, and in which eras did they dominate?
**Tables involved:** constructor_standings, races, constructors
**Key SQL / technique:** CTE, NATURAL JOIN, subquery on season-end standings, STRING_AGG

In [7]:
query_2 = r"""
WITH season_final_race AS (           
    SELECT year, MAX(round) AS round
    FROM races
    GROUP BY year
),
champions AS (                        
    SELECT year, "constructorId"
    FROM constructor_standings
    NATURAL JOIN races                
    NATURAL JOIN season_final_race    
    WHERE position = 1
)
SELECT c.name                                              AS constructor,
       COUNT(*)                                            AS championships,
       MIN(ch.year)                                        AS first_title,
       MAX(ch.year)                                        AS latest_title,
       STRING_AGG(ch.year::text, ', ' ORDER BY ch.year)    AS title_years
FROM champions ch
NATURAL JOIN constructors c           
GROUP BY c.name
ORDER BY championships DESC, first_title;
"""

query_2_df = run_query(query_2)

Rows returned: 17

     constructor  championships  first_title  latest_title                                                                      title_years
0        Ferrari             16         1961          2008  1961, 1964, 1975, 1976, 1977, 1979, 1982, 1983, 1999, 2000, 2001, 2002, 2003...
1        McLaren              9         1974          2024                             1974, 1984, 1985, 1988, 1989, 1990, 1991, 1998, 2024
2       Williams              9         1980          1997                             1980, 1981, 1986, 1987, 1992, 1993, 1994, 1996, 1997
3       Mercedes              8         2014          2021                                   2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021
4       Red Bull              6         2010          2023                                               2010, 2011, 2012, 2013, 2022, 2023
5     Team Lotus              4         1970          1978                                                           1970, 1972, 1973, 1978
6

## Query 3: All-time points leaders

**Question:** Who are the top 10 drivers by total career championship points?
**Tables involved:** drivers, driver_standings, races
**Key SQL / technique:** CTE, NATURAL JOIN, GROUP BY, SUM / MAX

In [8]:
query_3 = r"""
WITH season_final_race AS (           
    SELECT year, MAX(round) AS round
    FROM races
    GROUP BY year
),
season_end AS (                       
    SELECT "driverId", year, points, position
    FROM driver_standings
    NATURAL JOIN races                
    NATURAL JOIN season_final_race    
)
SELECT d.forename || ' ' || d.surname                     AS driver,
       SUM(se.points)                                     AS career_points,
       COUNT(*)                                           AS seasons,
       COUNT(*) FILTER (WHERE se.position = 1)            AS world_titles,
       MAX(se.points)                                     AS best_season_points,
       ROUND((SUM(se.points) / COUNT(*))::numeric, 1)     AS points_per_season
FROM season_end se
NATURAL JOIN drivers d                
GROUP BY "driverId", d.forename, d.surname
ORDER BY career_points DESC
LIMIT 10;
"""

query_3_df = run_query(query_3)

Rows returned: 10

               driver  career_points  seasons  world_titles  best_season_points  points_per_season
0      Lewis Hamilton         4862.5       18             7               413.0              270.1
1    Sebastian Vettel         3098.0       16             4               397.0              193.6
2      Max Verstappen         3023.5       10             4               575.0              302.4
3     Fernando Alonso         2337.0       21             2               278.0              111.3
4      Kimi Räikkönen         1873.0       19             1               251.0               98.6
5     Valtteri Bottas         1797.0       12             0               326.0              149.8
6        Sergio Pérez         1638.0       14             0               305.0              117.0
7        Nico Rosberg         1594.5       11             1               385.0              145.0
8  Michael Schumacher         1566.0       19             7               148.0           

## Query 4: Pit-stop efficiency

**Question:** What is the average pit-stop duration per constructor per season, and which team is consistently the fastest in the pits?
**Tables involved:** pit_stops, results, races, constructors
**Key SQL / technique:** CTE, NATURAL JOIN, AVG, GROUP BY, window function RANK()

In [9]:
query_4 = r"""
WITH stops AS (                       
    SELECT year, "constructorId", milliseconds
    FROM pit_stops
    NATURAL JOIN results             
    NATURAL JOIN races                
    WHERE milliseconds < 60000        
),
season_avg AS (
    SELECT year,
           "constructorId",
           COUNT(*)                                                   AS stops,
           AVG(milliseconds) / 1000.0                                 AS avg_stop_s,
           RANK() OVER (PARTITION BY year ORDER BY AVG(milliseconds)) AS season_rank
    FROM stops
    GROUP BY year, "constructorId"
    HAVING COUNT(*) >= 20             
)
SELECT c.name                                          AS constructor,
       COUNT(*)                                        AS seasons,
       COUNT(*) FILTER (WHERE sa.season_rank = 1)      AS seasons_fastest,
       ROUND(AVG(sa.season_rank), 1)                   AS avg_season_rank,
       ROUND(MIN(sa.avg_stop_s), 2)                    AS best_season_avg_s,
       ROUND(AVG(sa.avg_stop_s), 2)                    AS overall_avg_s
FROM season_avg sa
NATURAL JOIN constructors c           
GROUP BY c.name
HAVING COUNT(*) >= 3
ORDER BY seasons_fastest DESC, avg_season_rank;
"""

query_4_df = run_query(query_4)

Rows returned: 17

      constructor  seasons  seasons_fastest  avg_season_rank  best_season_avg_s  overall_avg_s
0        Red Bull       14                7              2.3              22.15          23.75
1        Mercedes       14                4              2.5              22.46          23.77
2         Ferrari       14                2              3.4              22.24          23.97
3         McLaren       14                1              5.0              22.58          24.31
4    Aston Martin        4                0              4.8              24.13          24.55
5  Alpine F1 Team        4                0              4.8              24.00          24.78
6         Renault        6                0              5.3              23.24          24.37
7     Force India        8                0              6.0              22.69          24.16
8      AlphaTauri        4                0              6.0              24.54          25.02
9        Williams       14     

## Query 5: Busiest circuits

**Question:** Which circuits have hosted the highest number of Grands Prix?
**Tables involved:** circuits, races
**Key SQL / technique:** JOIN ... ON, COUNT, GROUP BY

In [10]:
query_5 = r"""
SELECT ci.name             AS circuit,
       ci.location,
       ci.country,
       COUNT(*)            AS grands_prix,
       MIN(ra.year)        AS first_race,
       MAX(ra.year)        AS latest_race
FROM races ra
JOIN circuits ci ON ci."circuitId" = ra."circuitId"
GROUP BY ci."circuitId", ci.name, ci.location, ci.country
ORDER BY grands_prix DESC, first_race;
"""

query_5_df = run_query(query_5)

Rows returned: 77

                        circuit     location  country  grands_prix  first_race  latest_race
0  Autodromo Nazionale di Monza        Monza    Italy           74        1950         2024
1             Circuit de Monaco  Monte-Carlo   Monaco           70        1950         2024
2           Silverstone Circuit  Silverstone       UK           59        1950         2024
3  Circuit de Spa-Francorchamps          Spa  Belgium           57        1950         2024
4     Circuit Gilles Villeneuve     Montreal   Canada           43        1978         2024
5                   Nürburgring      Nürburg  Germany           41        1951         2020
6    Autódromo José Carlos Pace    São Paulo   Brazil           41        1973         2024
7                   Hungaroring     Budapest  Hungary           39        1986         2024
8                 Red Bull Ring    Spielberg  Austria           38        1970         2024
9                Hockenheimring   Hockenheim  Germany        

## Query 5b: Busiest circuits (part b)

**Question:** How has the race calendar evolved geographically, decade by decade?
**Tables involved:** circuits, races
**Key SQL / technique:** JOIN ... ON, derived decade column, COUNT(DISTINCT), GROUP BY

In [11]:
query_5b = r"""
SELECT (ra.year / 10) * 10                                     AS decade,
       COUNT(*)                                                AS races,
       COUNT(DISTINCT ra."circuitId")                          AS circuits_used,
       COUNT(DISTINCT ci.country)                              AS countries,
       ROUND(COUNT(*)::numeric / COUNT(DISTINCT ra.year), 1)   AS races_per_season
FROM races ra
JOIN circuits ci ON ci."circuitId" = ra."circuitId"     
GROUP BY decade
ORDER BY decade;
"""

query_5b_df = run_query(query_5b)

Rows returned: 8

   decade  races  circuits_used  countries  races_per_season
0    1950     84             19         13               8.4
1    1960    100             25         15              10.0
2    1970    144             28         17              14.4
3    1980    156             31         19              15.6
4    1990    162             26         19              16.2
5    2000    174             24         20              17.4
6    2010    198             27         25              19.8
7    2020    107             30         27              21.4


## Query 6: Lap-time evolution

**Question:** How has the lap time at a given circuit (Monza) changed over the decades?
**Tables involved:** lap_times, races, circuits
**Key SQL / technique:** NATURAL JOIN, JOIN ... ON, AVG, MIN, PERCENTILE_CONT, GROUP BY (decade bucket)

In [12]:
query_6 = r"""
SELECT (ra.year / 10) * 10                                     AS decade,
       COUNT(DISTINCT "raceId")                                AS races,
       COUNT(*)                                                AS laps_recorded,
       ROUND(AVG(lt.milliseconds) / 1000.0, 3)                 AS avg_lap_s,
       ROUND((PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY lt.milliseconds)
              / 1000.0)::numeric, 3)                           AS median_lap_s,
       ROUND(MIN(lt.milliseconds) / 1000.0, 3)                 AS fastest_lap_s
FROM lap_times lt
NATURAL JOIN races ra                                   
JOIN circuits ci ON ci."circuitId" = ra."circuitId"     
WHERE ci."circuitRef" = 'monza'
GROUP BY decade
ORDER BY decade;
"""

query_6_df = run_query(query_6)

Rows returned: 4

   decade  races  laps_recorded  avg_lap_s  median_lap_s  fastest_lap_s
0    1990      4           3262     89.345        87.893         84.808
1    2000     10           9109     90.225        86.521         81.046
2    2010     10          10207     90.006        88.827         81.779
3    2020      5           4745     95.521        86.667         81.432


## Query 7: Qualifying vs. race outcome

**Question:** What is the correlation between a driver's qualifying position and their final race position?
**Tables involved:** qualifying, results, races
**Key SQL / technique:** JOIN ... ON (raceId, driverId), CORR(), FILTER, GROUP BY

In [13]:
query_7 = r"""
SELECT ra.year,
       COUNT(DISTINCT ra."raceId")                                          AS races,
       COUNT(*)                                                             AS driver_results,
       ROUND(CORR(q.position, NULLIF(r.position, '\N')::int)::numeric, 3)   AS corr_quali_vs_finish,
       COUNT(*) FILTER (WHERE q.position = 1
                          AND NULLIF(r.position, '\N')::int = 1)            AS wins_from_pole,
       ROUND(100.0 * COUNT(*) FILTER (WHERE q.position = 1
                                        AND NULLIF(r.position, '\N')::int = 1)
             / COUNT(DISTINCT ra."raceId"), 1)                              AS pct_races_won_from_pole
FROM qualifying q
JOIN results r ON r."raceId" = q."raceId" AND r."driverId" = q."driverId"
JOIN races ra  ON ra."raceId" = q."raceId"
WHERE NULLIF(r.position, '\N')::int IS NOT NULL          
GROUP BY ra.year
ORDER BY ra.year DESC;
"""

query_7_df = run_query(query_7)

Rows returned: 31

   year  races  driver_results  corr_quali_vs_finish  wins_from_pole  pct_races_won_from_pole
0  2024     24             431                 0.803              12                     50.0
1  2023     22             386                 0.698              15                     68.2
2  2022     22             376                 0.749              10                     45.5
3  2021     22             387                 0.759              12                     54.5
4  2020     17             287                 0.749              10                     58.8
5  2019     21             368                 0.736               8                     38.1
6  2018     21             340                 0.799              10                     47.6
7  2017     20             313                 0.767              11                     55.0
8  2016     21             379                 0.799              13                     61.9
9  2015     19             303           

## Query 8: Winners by nationality

**Question:** What is the nationality distribution of race winners, and which countries have produced the most successful drivers?
**Tables involved:** drivers, results
**Key SQL / technique:** NATURAL JOIN, COUNT, GROUP BY, window function SUM() OVER ()

In [14]:
query_8 = r"""
SELECT d.nationality,
       COUNT(*)                                              AS race_wins,
       COUNT(DISTINCT "driverId")                            AS different_winners,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)    AS pct_of_all_wins
FROM results r
NATURAL JOIN drivers d                      
WHERE NULLIF(r.position, '\N')::int = 1
GROUP BY d.nationality
ORDER BY race_wins DESC;
"""

query_8_df = run_query(query_8)

Rows returned: 23

  nationality  race_wins  different_winners  pct_of_all_wins
0     British        317                 21             28.1
1      German        179                  7             15.9
2   Brazilian        101                  6              9.0
3      French         81                 14              7.2
4       Dutch         63                  1              5.6
5     Finnish         57                  5              5.1
6  Australian         45                  5              4.0
7     Italian         43                 15              3.8
8    Austrian         41                  3              3.6
9   Argentine         38                  3              3.4


## Query 9: DNF / retirement analysis

**Question:** Which constructors have the highest number of race retirements (DNFs), and what are the most common causes?
**Tables involved:** status, results, constructors
**Key SQL / technique:** CTE, NATURAL JOIN, CASE, FILTER, MODE()

In [15]:
query_9 = r"""
WITH dnf AS (                         
    SELECT "constructorId",
           status,
           CASE
               WHEN status ILIKE ANY (ARRAY['%engine%', '%power unit%', '%turbo%', '%overheating%',
                                            '%oil%', '%water%', '%radiator%', '%ers%', '%fuel%',
                                            '%ignition%', '%injection%', '%exhaust%', '%alternator%',
                                            '%magneto%', '%throttle%'])
                    THEN 'Engine / power unit'
               WHEN status ILIKE ANY (ARRAY['%accident%', '%collision%', '%spun off%', '%damage%'])
                    THEN 'Accident / collision'
               WHEN status ILIKE ANY (ARRAY['%gearbox%', '%transmission%', '%clutch%', '%driveshaft%',
                                            '%halfshaft%', '%differential%', '%axle%'])
                    THEN 'Gearbox / transmission'
               ELSE 'Other'
           END AS cause_group
    FROM results
    NATURAL JOIN status               
    WHERE status <> 'Finished'
      AND status NOT LIKE '+%'        
      AND status NOT IN ('Did not qualify', 'Did not prequalify', 'Withdrew',
                         'Did not start', 'Disqualified', 'Excluded', '107% Rule')
)
SELECT c.name                                                          AS constructor,
       COUNT(*)                                                        AS dnfs,
       COUNT(*) FILTER (WHERE cause_group = 'Engine / power unit')     AS engine,
       COUNT(*) FILTER (WHERE cause_group = 'Accident / collision')    AS accident,
       COUNT(*) FILTER (WHERE cause_group = 'Gearbox / transmission')  AS gearbox,
       COUNT(*) FILTER (WHERE cause_group = 'Other')                   AS other,
       MODE() WITHIN GROUP (ORDER BY status)                           AS most_common_cause
FROM dnf
NATURAL JOIN constructors c           
GROUP BY c.name
ORDER BY dnfs DESC;
"""

query_9_df = run_query(query_9)

Rows returned: 185

  constructor  dnfs  engine  accident  gearbox  other most_common_cause
0     Ferrari   708     279       203      102    124            Engine
1     McLaren   552     193       188       62    109            Engine
2    Williams   486     137       184       56    109            Engine
3  Team Lotus   416     152       108       76     80            Engine
4     Tyrrell   382     136       137       60     49            Engine
5     Minardi   351      97       109       82     63            Engine
6     Brabham   331     145        68       53     65            Engine
7      Ligier   285     107        91       36     51            Engine
8      Arrows   282     100        79       39     64            Engine
9         BRM   271     146        43       44     38            Engine


## Query 10: Season points progression

**Question:** How do the championship points of the top 5 drivers of the 2021 season progress race by race?
**Tables involved:** driver_standings, races, drivers
**Key SQL / technique:** CTE, NATURAL JOIN, window functions LAG(), MAX() OVER

In [16]:
query_10 = r"""
WITH season_races AS (                
    SELECT "raceId", round, name
    FROM races
    WHERE year = 2021
),
top5 AS (                             
    SELECT "driverId"
    FROM driver_standings
    WHERE "raceId" = (SELECT "raceId" FROM season_races ORDER BY round DESC LIMIT 1)
      AND position <= 5
)
SELECT sr.round,
       sr.name                                                     AS race,
       d.surname                                                   AS driver,
       ds.points                                                   AS total_points,
       ds.points - COALESCE(LAG(ds.points, 1)
                   OVER (PARTITION BY "driverId" ORDER BY sr.round), 0) AS points_this_race,
       MAX(ds.points) OVER (PARTITION BY sr.round) - ds.points     AS gap_to_leader,
       ds.position                                                 AS championship_position
FROM driver_standings ds
NATURAL JOIN season_races sr          
NATURAL JOIN top5                     
NATURAL JOIN drivers d                
ORDER BY sr.round, ds.position;
"""

query_10_df = run_query(query_10)

Rows returned: 110

   round                       race      driver  total_points  points_this_race  gap_to_leader  championship_position
0      1         Bahrain Grand Prix    Hamilton          25.0              25.0            0.0                      1
1      1         Bahrain Grand Prix  Verstappen          18.0              18.0            7.0                      2
2      1         Bahrain Grand Prix      Bottas          16.0              16.0            9.0                      3
3      1         Bahrain Grand Prix       Pérez          10.0              10.0           15.0                      5
4      1         Bahrain Grand Prix       Sainz           4.0               4.0           21.0                      8
5      2  Emilia Romagna Grand Prix    Hamilton          44.0              19.0            0.0                      1
6      2  Emilia Romagna Grand Prix  Verstappen          43.0              25.0            1.0                      2
7      2  Emilia Romagna Grand Prix 

## Query 11: Fastest-lap leaders (drivers)

**Question:** Which drivers hold the most 'fastest lap of the race' awards?
**Tables involved:** results, drivers, races
**Key SQL / technique:** NATURAL JOIN, filter on rank = 1, GROUP BY

In [17]:
query_11 = r"""
SELECT d.forename || ' ' || d.surname        AS driver,
       COUNT(*)                              AS fastest_laps,
       COUNT(DISTINCT ra.year)               AS seasons_with_fastest_lap,
       MIN(ra.year)                          AS first,
       MAX(ra.year)                          AS latest
FROM results r
NATURAL JOIN drivers d                
NATURAL JOIN races ra                 
WHERE r.rank = '1'                    
GROUP BY "driverId", d.forename, d.surname
ORDER BY fastest_laps DESC;
"""

query_11_df = run_query(query_11)

Rows returned: 41

               driver  fastest_laps  seasons_with_fastest_lap  first  latest
0      Lewis Hamilton            66                        17   2007    2024
1      Kimi Räikkönen            42                        12   2004    2018
2    Sebastian Vettel            38                        11   2009    2019
3      Max Verstappen            33                         9   2016    2024
4     Fernando Alonso            25                        11   2005    2024
5  Michael Schumacher            21                         4   2004    2012
6        Nico Rosberg            20                         6   2006    2016
7     Valtteri Bottas            19                         6   2014    2021
8         Mark Webber            19                         5   2009    2013
9    Daniel Ricciardo            17                         8   2014    2024


## Query 11b: Fastest-lap leaders (constructors)

**Question:** Which constructors hold the most fastest-lap awards, and how are these distributed across seasons?
**Tables involved:** results, constructors, races
**Key SQL / technique:** NATURAL JOIN, JOIN ... ON, filter on rank = 1, GROUP BY, window function

In [18]:
query_11b = r"""
SELECT c.name                                                   AS constructor,
       COUNT(*)                                                 AS fastest_laps,
       COUNT(DISTINCT ra.year)                                  AS seasons,
       ROUND(COUNT(*)::numeric / COUNT(DISTINCT ra.year), 1)    AS per_season,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)       AS pct_of_all
FROM results r
NATURAL JOIN races ra                                        
JOIN constructors c ON c."constructorId" = r."constructorId" 
WHERE r.rank = '1'
GROUP BY c.name
ORDER BY fastest_laps DESC;
"""

query_11b_df = run_query(query_11b)

Rows returned: 19

    constructor  fastest_laps  seasons  per_season  pct_of_all
0      Red Bull            99       16         6.2        24.1
1      Mercedes            99       13         7.6        24.1
2       Ferrari            96       17         5.6        23.4
3       McLaren            60       16         3.8        14.6
4       Renault            14        5         2.8         3.4
5      Williams             7        5         1.4         1.7
6      Lotus F1             5        2         2.5         1.2
7   Force India             5        5         1.0         1.2
8         Brawn             4        1         4.0         1.0
9  Haas F1 Team             3        3         1.0         0.7


## Query 12: Home-race advantage

**Question:** Do drivers perform better (higher finishing position) at their home Grand Prix compared to their season average?
**Tables involved:** drivers, races, circuits, results
**Key SQL / technique:** CTE, NATURAL JOIN, CASE, self-referencing comparison, AVG

In [19]:
query_12 = r"""
WITH nationality_country(nationality, country) AS (
    VALUES ('British', 'UK'),            ('American', 'USA'),          ('American', 'United States'),
           ('Italian', 'Italy'),         ('French', 'France'),         ('German', 'Germany'),
           ('Brazilian', 'Brazil'),      ('Argentine', 'Argentina'),   ('Belgian', 'Belgium'),
           ('South African', 'South Africa'), ('Swiss', 'Switzerland'), ('Japanese', 'Japan'),
           ('Australian', 'Australia'),  ('Dutch', 'Netherlands'),     ('Spanish', 'Spain'),
           ('Austrian', 'Austria'),      ('Canadian', 'Canada'),       ('Swedish', 'Sweden'),
           ('Mexican', 'Mexico'),        ('Monegasque', 'Monaco'),     ('Portuguese', 'Portugal'),
           ('Russian', 'Russia'),        ('Chinese', 'China'),         ('Indian', 'India'),
           ('Malaysian', 'Malaysia'),    ('Hungarian', 'Hungary')
),
entries AS (                          
    SELECT "driverId",
           ra.year,
           r."positionOrder",
           CASE WHEN nc.country IS NOT NULL THEN 1 ELSE 0 END AS is_home
    FROM results r
    NATURAL JOIN races ra                                   
    NATURAL JOIN drivers d                                  
    JOIN circuits ci ON ci."circuitId" = ra."circuitId"     
    LEFT JOIN nationality_country nc
           ON nc.nationality = d.nationality
          AND nc.country = ci.country
),
per_season AS (                       
    SELECT "driverId",
           year,
           COUNT(*) FILTER (WHERE is_home = 1)                AS home_races,
           AVG("positionOrder") FILTER (WHERE is_home = 1)    AS avg_home,
           AVG("positionOrder") FILTER (WHERE is_home = 0)    AS avg_other
    FROM entries
    GROUP BY "driverId", year
    HAVING COUNT(*) FILTER (WHERE is_home = 1) > 0
       AND COUNT(*) FILTER (WHERE is_home = 0) > 0
)
SELECT d.forename || ' ' || d.surname                  AS driver,
       d.nationality,
       SUM(ps.home_races)::int                         AS home_races,
       ROUND(AVG(ps.avg_home), 2)                      AS avg_finish_home,
       ROUND(AVG(ps.avg_other), 2)                     AS avg_finish_other,
       ROUND(AVG(ps.avg_other - ps.avg_home), 2)       AS places_gained_at_home,
       CASE WHEN AVG(ps.avg_other - ps.avg_home) > 0
            THEN 'Better at home'
            ELSE 'Not better at home'
       END                                             AS verdict
FROM per_season ps
NATURAL JOIN drivers d                
GROUP BY "driverId", d.forename, d.surname, d.nationality
HAVING SUM(ps.home_races) >= 8
ORDER BY home_races DESC, driver;
"""

query_12_df = run_query(query_12)

Rows returned: 71

                 driver nationality  home_races  avg_finish_home  avg_finish_other  places_gained_at_home             verdict
0      Riccardo Patrese     Italian          29            13.91             12.66                  -1.25  Not better at home
1    Michael Schumacher      German          28             5.35              6.72                   1.37      Better at home
2      Michele Alboreto     Italian          28            15.46             14.41                  -1.06  Not better at home
3     Andrea de Cesaris     Italian          27            14.96             16.14                   1.18      Better at home
4       Fernando Alonso     Spanish          26             8.05              8.48                   0.43      Better at home
5          Jarno Trulli     Italian          25            13.43             12.17                  -1.26  Not better at home
6  Giancarlo Fisichella     Italian          24            11.79             11.11                 

## Query 12b: Home-race advantage (overall)

**Question:** Across all drivers, is the average finishing position better at home races?
**Tables involved:** drivers, races, circuits, results
**Key SQL / technique:** CTE, NATURAL JOIN, CASE, AVG, conditional aggregation

In [20]:
query_12b = r"""
WITH nationality_country(nationality, country) AS (
    VALUES ('British', 'UK'),            ('American', 'USA'),          ('American', 'United States'),
           ('Italian', 'Italy'),         ('French', 'France'),         ('German', 'Germany'),
           ('Brazilian', 'Brazil'),      ('Argentine', 'Argentina'),   ('Belgian', 'Belgium'),
           ('South African', 'South Africa'), ('Swiss', 'Switzerland'), ('Japanese', 'Japan'),
           ('Australian', 'Australia'),  ('Dutch', 'Netherlands'),     ('Spanish', 'Spain'),
           ('Austrian', 'Austria'),      ('Canadian', 'Canada'),       ('Swedish', 'Sweden'),
           ('Mexican', 'Mexico'),        ('Monegasque', 'Monaco'),     ('Portuguese', 'Portugal'),
           ('Russian', 'Russia'),        ('Chinese', 'China'),         ('Indian', 'India'),
           ('Malaysian', 'Malaysia'),    ('Hungarian', 'Hungary')
),
entries AS (                          
    SELECT "driverId",
           ra.year,
           r."positionOrder",
           CASE WHEN nc.country IS NOT NULL THEN 1 ELSE 0 END AS is_home
    FROM results r
    NATURAL JOIN races ra                                   
    NATURAL JOIN drivers d                                  
    JOIN circuits ci ON ci."circuitId" = ra."circuitId"     
    LEFT JOIN nationality_country nc
           ON nc.nationality = d.nationality
          AND nc.country = ci.country
),
per_season AS (                       
    SELECT "driverId",
           year,
           COUNT(*) FILTER (WHERE is_home = 1)                AS home_races,
           AVG("positionOrder") FILTER (WHERE is_home = 1)    AS avg_home,
           AVG("positionOrder") FILTER (WHERE is_home = 0)    AS avg_other
    FROM entries
    GROUP BY "driverId", year
    HAVING COUNT(*) FILTER (WHERE is_home = 1) > 0
       AND COUNT(*) FILTER (WHERE is_home = 0) > 0
)
SELECT COUNT(*)                                       AS driver_seasons_compared,
       ROUND(AVG(avg_home), 2)                        AS avg_finish_home,
       ROUND(AVG(avg_other), 2)                       AS avg_finish_other,
       ROUND(AVG(avg_other - avg_home), 2)            AS avg_places_gained_at_home,
       ROUND(100.0 * COUNT(*) FILTER (WHERE avg_home < avg_other)
             / COUNT(*), 1)                           AS pct_seasons_better_at_home
FROM per_season;
"""

query_12b_df = run_query(query_12b)

Rows returned: 1

   driver_seasons_compared  avg_finish_home  avg_finish_other  avg_places_gained_at_home  pct_seasons_better_at_home
0                     1538            12.82             13.12                        0.3                        54.7


## Query 13: Decade-wise team dominance

**Question:** What share of total points did each constructor earn per decade (top 3 per decade)?
**Tables involved:** constructor_standings, races, constructors
**Key SQL / technique:** CTE, NATURAL JOIN, GROUP BY, window functions SUM() OVER (% share), RANK()

In [21]:
query_13 = r"""
WITH season_final_race AS (           
    SELECT year, MAX(round) AS round
    FROM races
    GROUP BY year
),
season_points AS (                    
    SELECT year, "constructorId", points
    FROM constructor_standings
    NATURAL JOIN races                
    NATURAL JOIN season_final_race    
),
decade_points AS (
    SELECT (year / 10) * 10    AS decade,
           "constructorId",
           SUM(points)         AS points
    FROM season_points
    GROUP BY decade, "constructorId"
),
ranked AS (
    SELECT decade,
           "constructorId",
           points,
           ROUND((100.0 * points / SUM(points) OVER (PARTITION BY decade))::numeric, 1) AS pct_share,
           RANK() OVER (PARTITION BY decade ORDER BY points DESC)                     AS rank_in_decade
    FROM decade_points
)
SELECT r.decade,
       r.rank_in_decade,
       c.name          AS constructor,
       r.points,
       r.pct_share
FROM ranked r
NATURAL JOIN constructors c           
WHERE r.rank_in_decade <= 3
ORDER BY r.decade, r.rank_in_decade;
"""

query_13_df = run_query(query_13)

Rows returned: 24

   decade  rank_in_decade    constructor  points  pct_share
0    1950               1        Ferrari    72.0       29.9
1    1950               2        Vanwall    48.0       19.9
2    1950               3  Cooper-Climax    40.0       16.6
3    1960               1        Ferrari   276.0       15.1
4    1960               2            BRM   254.0       13.9
5    1960               3   Lotus-Climax   224.0       12.3
6    1970               1        Ferrari   616.5       19.7
7    1970               2     Team Lotus   479.0       15.3
8    1970               3        Tyrrell   447.0       14.3
9    1980               1        McLaren   887.5       22.9


## Query 14: Grid position vs. points scored

**Question:** Does starting grid position show a meaningful relationship with points scored in the race (2010 onward, same points system)?
**Tables involved:** results, races
**Key SQL / technique:** NATURAL JOIN, CASE (grid buckets), AVG, FILTER, GROUP BY

In [22]:
query_14 = r"""
SELECT CASE
           WHEN r.grid = 0               THEN 'Pit lane'
           WHEN r.grid = 1               THEN 'P1 (pole)'
           WHEN r.grid BETWEEN 2 AND 3   THEN 'P2-P3'
           WHEN r.grid BETWEEN 4 AND 6   THEN 'P4-P6'
           WHEN r.grid BETWEEN 7 AND 10  THEN 'P7-P10'
           WHEN r.grid BETWEEN 11 AND 15 THEN 'P11-P15'
           ELSE 'P16+'
       END                                                          AS grid_bucket,
       COUNT(*)                                                     AS starts,
       ROUND(AVG(r.points)::numeric, 2)                             AS avg_points,
       ROUND(100.0 * COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int = 1)
             / COUNT(*), 1)                                         AS win_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int <= 3)
             / COUNT(*), 1)                                         AS podium_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE r.points > 0)
             / COUNT(*), 1)                                         AS points_finish_pct
FROM results r
NATURAL JOIN races ra                 
WHERE ra.year >= 2010                 
GROUP BY grid_bucket
ORDER BY MIN(CASE WHEN r.grid = 0 THEN 99 ELSE r.grid END);
"""

query_14_df = run_query(query_14)

Rows returned: 7

  grid_bucket  starts  avg_points  win_pct  podium_pct  points_finish_pct
0   P1 (pole)     305       18.70     51.1        80.0               90.5
1       P2-P3     608       14.06     18.3        59.9               88.0
2       P4-P6     915        8.81      2.6        23.0               80.1
3      P7-P10    1216        4.13      0.8         4.9               62.5
4     P11-P15    1518        1.65      0.2         1.6               33.7
5        P16+    1789        0.60      0.1         0.8               12.3
6    Pit lane      85        0.61      0.0         0.0               16.5


## Query 15: Sprint vs. Grand Prix performance

**Question:** For seasons with sprint races (2021 onward), how does a driver's sprint result compare with their main-race result on the same weekend?
**Tables involved:** sprint_results, results, races, drivers
**Key SQL / technique:** JOIN ... ON (raceId, driverId), AVG, comparative aggregation

In [23]:
query_15 = r"""
SELECT d.forename || ' ' || d.surname                                AS driver,
       COUNT(*)                                                      AS sprint_weekends,
       ROUND(AVG(s."positionOrder"), 2)                              AS avg_sprint_finish,
       ROUND(AVG(r."positionOrder"), 2)                              AS avg_gp_finish,
       ROUND(AVG(r."positionOrder" - s."positionOrder"), 2)          AS gp_minus_sprint,
       SUM(s.points)                                                 AS sprint_points,
       SUM(r.points)                                                 AS gp_points_same_weekends,
       COUNT(*) FILTER (WHERE s."positionOrder" = r."positionOrder") AS same_position_both
FROM sprint_results s
JOIN results r ON r."raceId" = s."raceId" AND r."driverId" = s."driverId"
JOIN drivers d ON d."driverId" = s."driverId"
GROUP BY d."driverId", d.forename, d.surname
HAVING COUNT(*) >= 5
ORDER BY sprint_points DESC;
"""

query_15_df = run_query(query_15)

Rows returned: 23

            driver  sprint_weekends  avg_sprint_finish  avg_gp_finish  gp_minus_sprint  sprint_points  gp_points_same_weekends  same_position_both
0   Max Verstappen               18               2.00           3.83             1.83          111.0                    335.0                   7
1     Carlos Sainz               18               4.89           8.61             3.72           69.0                    141.0                   4
2  Charles Leclerc               18               5.06           5.61             0.56           67.0                    223.0                   1
3     Lando Norris               18               6.33           6.33             0.00           57.0                    189.0                   2
4     Sergio Pérez               18               8.06           7.06            -1.00           53.0                    169.0                   3
5   George Russell               18               7.28           6.50            -0.78           50

In [24]:
engine.dispose()
print("Database connection closed.")

Database connection closed.
